# 01 · Intent datasets and fixed 8-shot splits

Prepare the BANKING77 and in-domain CLINC150 splits used by all baselines. The final table reports sample counts, class balance and utterance length. Run 02 for the SetFit pilot.

In [1]:
import csv
import hashlib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from datasets import load_dataset
from IPython.display import display
from sklearn.model_selection import train_test_split

SEED = 42
K = 8
VALIDATION = 0.10

cwd = Path.cwd()
PROJECT = next((p for p in (cwd, *cwd.parents) if (p / 'notebooks/01_dataset_exploration.ipynb').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Open this notebook from the project or its notebooks directory.')
OUT = PROJECT / 'results/protocol'
tables = {}
metadata_files = {}
# Download the reviewed original data when starting from a fresh clone.
clinc_file = PROJECT / 'data/clinc150/data_full.json'
if not clinc_file.exists():
    import urllib.request
    content = urllib.request.urlopen('https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json', timeout=90).read()
    expected_hash = '36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0'
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError('Original CLINC download differs from the reviewed version')
    clinc_file.parent.mkdir(parents=True, exist_ok=True)
    pending = clinc_file.with_suffix('.download')
    pending.write_bytes(content)
    pending.replace(clinc_file)


C:\APP\Anaconda\envs\py3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Text overlap

Case-fold text, trim the ends and collapse whitespace. Keep punctuation. Remove matching rows from the training side, including matches with different labels, and log both original labels. Report unique texts separately from matching row pairs.

In [2]:
def text_key(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError('Text must be a non-empty string')
    return re.sub(r'\s+', ' ', text).strip().casefold()

def text_hash(text):
    return hashlib.sha256(text_key(text).encode('utf-8')).hexdigest()

def audit(left, right, left_name, right_name):
    a = left.assign(_key=left['text'].map(text_key))
    b = right.assign(_key=right['text'].map(text_key))
    pairs = a.merge(b, on='_key', suffixes=('_left', '_right'))
    conflicts = pairs[pairs['label_left'] != pairs['label_right']]
    report = {'left': left_name, 'right': right_name, 'matching_pairs': len(pairs),
              'unique_texts': pairs['_key'].nunique(),
              'left_rows': pairs['source_id_left'].nunique(),
              'right_rows': pairs['source_id_right'].nunique(), 'conflicting_pairs': len(conflicts)}
    return report, pairs

def remove_train_overlap(train, protected, name):
    report, pairs = audit(train, protected, name, 'protected')
    print('Before removal:', report)
    if report['conflicting_pairs']:
        display(pairs.loc[pairs.label_left != pairs.label_right, ['text_left','label_left','text_right','label_right']].head(20))
        print('Conflicting labels are recorded; training-side rows are removed under the stated duplicate policy.')
    drop_ids = set(pairs['source_id_left'])
    clean = train.loc[~train.source_id.isin(drop_ids)].copy()
    print('Removed training rows:', len(train)-len(clean))
    return clean, report, pairs

def verify_disjoint(a, b, name_a, name_b):
    report, _ = audit(a, b, name_a, name_b)
    print(report)
    assert report['unique_texts'] == 0, f'Text overlap: {name_a} vs {name_b}'

def split_manifest(df, name):
    result = df[['source_id','label']].copy()
    result['text_sha256'] = df['text'].map(text_hash)
    tables[f'{name}.csv'] = result

## BANKING77

Remove training matches to official test, stratify the cleaned data with 10% held out, and remove any remaining training matches to validation. Draw eight examples from each training class using seed 42.

In [3]:
banking = load_dataset('PolyAI/banking77')
banking_labels = banking['train'].features['label']
assert banking_labels.num_classes == 77
bank_train = banking['train'].to_pandas()
bank_test = banking['test'].to_pandas()
for name, df in [('train', bank_train), ('test', bank_test)]:
    df['source_id'] = [f'banking77_{name}_{i}' for i in range(len(df))]
print('BANKING77 original rows:', len(bank_train), len(bank_test))
bank_clean, bank_test_report, bank_test_pairs = remove_train_overlap(bank_train, bank_test, 'banking_train')
tables['banking77_train_test_matches.csv'] = bank_test_pairs
bank_pool, bank_val = train_test_split(bank_clean, test_size=VALIDATION, random_state=SEED, stratify=bank_clean['label'])
bank_pool, bank_val_report, bank_val_pairs = remove_train_overlap(bank_pool, bank_val, 'banking_pool')
tables['banking77_pool_validation_matches.csv'] = bank_val_pairs
for a,b,an,bn in [(bank_pool,bank_val,'pool','validation'),(bank_pool,bank_test,'pool','test'),(bank_val,bank_test,'validation','test')]: verify_disjoint(a,b,an,bn)
assert bank_pool['label'].nunique() == bank_val['label'].nunique() == 77
assert bank_pool.groupby('label').size().min() >= K
fewshot_df = bank_pool.groupby('label', group_keys=False).sample(n=K, random_state=SEED).copy()
counts = fewshot_df.groupby('label').size()
assert len(fewshot_df) == 77*K and len(counts) == 77 and (counts == K).all()
print('BANKING77 final:', {'pool': len(bank_pool), 'validation': len(bank_val), 'test': len(bank_test), 'fewshot': len(fewshot_df), 'per_class_min': counts.min(), 'per_class_max': counts.max()})
for name,df in [('banking77_pool',bank_pool),('banking77_validation',bank_val),('banking77_test',bank_test),('banking77_8shot',fewshot_df)]: split_manifest(df,name)

Using the latest cached version of the dataset since PolyAI/banking77 couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at C:\Users\l_len\Documents\ChatGPT\UNNC_NLP\notebook_audit_tmp\datasets\PolyAI___banking77\default\0.0.0\1846984189f2bc0ac51c6fc3a600fa9e1714e0c6 (last modified on Tue Nov 10 17:16:17 2026).


BANKING77 original rows: 10003 3080
Before removal: {'left': 'banking_train', 'right': 'protected', 'matching_pairs': 7, 'unique_texts': 7, 'left_rows': 7, 'right_rows': 7, 'conflicting_pairs': 0}
Removed training rows: 7


Before removal: {'left': 'banking_pool', 'right': 'protected', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}
Removed training rows: 0
{'left': 'pool', 'right': 'validation', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}
{'left': 'pool', 'right': 'test', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}
{'left': 'validation', 'right': 'test', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}


BANKING77 final: {'pool': 8996, 'validation': 1000, 'test': 3080, 'fewshot': 616, 'per_class_min': np.int64(8), 'per_class_max': np.int64(8)}


## CLINC150 labels and out-of-scope records

Verify the 150 integer labels against the original named intents. The experiment uses the original in-domain splits. The cross-check below accounts for extra rows in the converted dataset. See [the provenance note](../references/clinc_provenance.md) for the upstream sources.

In [4]:
clinc_hf = load_dataset('DeepPavlov/clinc150', 'default')
clinc_path = PROJECT / 'data' / 'clinc150' / 'data_full.json'
original = json.loads(clinc_path.read_text(encoding='utf-8'))
print('Original CLINC JSON SHA256:', hashlib.sha256(clinc_path.read_bytes()).hexdigest())
hf_to_name = defaultdict(set)
clinc_raw = {}
for split, raw_name in [('train','train'), ('validation','val'), ('test','test')]:
    hf = clinc_hf[split].to_pandas()
    raw_in = original[raw_name]
    raw_oos = original['oos_' + raw_name]
    text_to_names = defaultdict(set)
    for text,label in raw_in: text_to_names[text].add(label)
    ambiguous = {text:names for text,names in text_to_names.items() if len(names)>1}
    assert not ambiguous, f'Ambiguous original labels in {split}: {len(ambiguous)}'
    known = hf[hf.label.notna()]
    assert Counter(known.utterance) == Counter(text for text, label in raw_in), f'In-domain sample mismatch: {split}'
    for text, label in zip(known.utterance, known.label):
        hf_to_name[int(label)].update(text_to_names[text])
    missing = Counter(hf.loc[hf.label.isna(), 'utterance'])
    oos = Counter(text for text,label in raw_oos)
    matched = missing & oos
    unknown = missing - oos
    unmatched_oos = oos - missing
    print(split, {'HF rows':len(hf), 'HF null labels':sum(missing.values()), 'original OOS matches':sum(matched.values()), 'null rows not in original OOS':sum(unknown.values()), 'original OOS not matched':sum(unmatched_oos.values()), 'original in-domain':len(raw_in)})
    if unknown:
        all_original = {text for rows in original.values() for text,label in rows}
        print('Unknown extra null rows also in any original split:', sum(n for text,n in unknown.items() if text in all_original))
    clinc_raw[split] = pd.DataFrame({'text':[x for x,y in raw_in], 'label':[y for x,y in raw_in], 'source_id':[f'clinc_{split}_{i}' for i in range(len(raw_in))]})
assert len(hf_to_name) == 150 and all(len(v)==1 for v in hf_to_name.values())
id_to_name = {int(k): next(iter(v)) for k,v in hf_to_name.items()}
assert len(set(id_to_name.values())) == 150
assert set(id_to_name.values()) == set(clinc_raw['train'].label)
for s in ['validation','test']: assert set(clinc_raw[s].label) == set(id_to_name.values())
metadata_files['clinc150_id_to_name.json'] = id_to_name
print('Verified CLINC mapping:', len(id_to_name), 'unique IDs and names')

Using the latest cached version of the dataset since DeepPavlov/clinc150 couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at C:\Users\l_len\Documents\ChatGPT\UNNC_NLP\notebook_audit_tmp\datasets\DeepPavlov___clinc150\default\0.0.0\d835118ecd5ffe5488d22e9e58d1c23d18c33229 (last modified on Tue Nov 10 17:15:18 2026).


Original CLINC JSON SHA256: 36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0


train {'HF rows': 15200, 'HF null labels': 200, 'original OOS matches': 100, 'null rows not in original OOS': 100, 'original OOS not matched': 0, 'original in-domain': 15000}
Unknown extra null rows also in any original split: 0


validation {'HF rows': 3100, 'HF null labels': 100, 'original OOS matches': 100, 'null rows not in original OOS': 0, 'original OOS not matched': 0, 'original in-domain': 3000}
test {'HF rows': 5500, 'HF null labels': 1000, 'original OOS matches': 1000, 'null rows not in original OOS': 0, 'original OOS not matched': 0, 'original in-domain': 4500}
Verified CLINC mapping: 150 unique IDs and names


## CLINC150 splits

Retain official validation and test. Remove training matches and record their labels before taking the fixed 8-shot sample.

In [5]:
clinc_train, clinc_val_report, clinc_val_pairs = remove_train_overlap(clinc_raw['train'], clinc_raw['validation'], 'clinc_train')
clinc_train, clinc_test_report, clinc_test_pairs = remove_train_overlap(clinc_train, clinc_raw['test'], 'clinc_train')
tables['clinc150_train_validation_matches.csv'] = clinc_val_pairs
tables['clinc150_train_test_matches.csv'] = clinc_test_pairs
for a,b,an,bn in [(clinc_train,clinc_raw['validation'],'train','validation'),(clinc_train,clinc_raw['test'],'train','test'),(clinc_raw['validation'],clinc_raw['test'],'validation','test')]: verify_disjoint(a,b,an,bn)
assert set(clinc_train.label) == set(id_to_name.values())
clinc_counts = clinc_train.groupby('label').size()
assert clinc_counts.min() >= K
clinc_fewshot = clinc_train.groupby('label', group_keys=False).sample(n=K, random_state=SEED).copy()
assert len(clinc_fewshot)==150*K and (clinc_fewshot.groupby('label').size()==K).all()
print('CLINC final:', {'train':len(clinc_train), 'validation':len(clinc_raw['validation']), 'test':len(clinc_raw['test']), 'classes':len(clinc_counts), '8shot':len(clinc_fewshot)})
for name,df in [('clinc150_pool',clinc_train),('clinc150_validation',clinc_raw['validation']),('clinc150_test',clinc_raw['test']),('clinc150_8shot',clinc_fewshot)]: split_manifest(df,name)
protocol = {'normalization':'strip/collapse whitespace and Unicode casefold; punctuation retained', 'seed':SEED, 'shots_per_class':K, 'banking_validation_fraction':VALIDATION, 'banking_train_vs_test':bank_test_report, 'banking_pool_vs_validation':bank_val_report, 'clinc_train_vs_validation':clinc_val_report, 'clinc_train_vs_test_after_validation':clinc_test_report, 'clinc_original_sha256':hashlib.sha256(clinc_path.read_bytes()).hexdigest(), 'note':'Review matched texts; reuse saved source IDs and text hashes across all baselines. CLINC unknown extra null rows excluded because original in-domain data is used.'}
metadata_files['protocol.json'] = protocol

Before removal: {'left': 'clinc_train', 'right': 'protected', 'matching_pairs': 3, 'unique_texts': 3, 'left_rows': 3, 'right_rows': 3, 'conflicting_pairs': 2}


,text_left,label_left,text_right,label_right
0,what is on my to do list,todo_list,what is on my to do list,reminder
1,turn up your volume,whisper_mode,turn up your volume,change_volume


Conflicting labels are recorded; training-side rows are removed under the stated duplicate policy.
Removed training rows: 3
Before removal: {'left': 'clinc_train', 'right': 'protected', 'matching_pairs': 2, 'unique_texts': 2, 'left_rows': 2, 'right_rows': 2, 'conflicting_pairs': 2}


,text_left,label_left,text_right,label_right
0,what's your designation,what_is_your_name,what's your designation,user_name
1,where did you grow up,how_old_are_you,where did you grow up,where_are_you_from


Conflicting labels are recorded; training-side rows are removed under the stated duplicate policy.
Removed training rows: 2
{'left': 'train', 'right': 'validation', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}


{'left': 'train', 'right': 'test', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}


{'left': 'validation', 'right': 'test', 'matching_pairs': 0, 'unique_texts': 0, 'left_rows': 0, 'right_rows': 0, 'conflicting_pairs': 0}
CLINC final: {'train': 14995, 'validation': 3000, 'test': 4500, 'classes': 150, '8shot': 1200}


## Data for the baseline comparison

The 8-shot rows are subsets of the corresponding training pools. Length is measured in whitespace-separated words.

In [6]:
rows = []
for dataset_name, parts in {
    'BANKING77': {'pool': bank_pool, 'validation': bank_val, 'test': bank_test, '8-shot': fewshot_df},
    'CLINC150': {'pool': clinc_train, 'validation': clinc_raw['validation'], 'test': clinc_raw['test'], '8-shot': clinc_fewshot},
}.items():
    for split_name, frame in parts.items():
        counts = frame['label'].value_counts()
        rows.append({'dataset': dataset_name, 'split': split_name, 'rows': len(frame),
                     'intents': len(counts), 'min per intent': counts.min(),
                     'max per intent': counts.max(), 'median words': frame['text'].str.split().str.len().median()})
display(pd.DataFrame(rows).set_index(['dataset', 'split']))

rows  intents  min per intent  max per intent  \
dataset   split                                                        
BANKING77 pool         8996       77              32             168   
          validation   1000       77               3              19   
          test         3080       77              40              40   
          8-shot        616       77               8               8   
CLINC150  pool        14995      150              99             100   
          validation   3000      150              20              20   
          test         4500      150              30              30   
          8-shot       1200      150               8               8   

                      median words  
dataset   split                     
BANKING77 pool                10.0  
          validation          10.0  
          test                 9.0  
          8-shot              10.0  
CLINC150  pool                 8.0  
          validation           8.0  
          test                 8.0  
          8-shot               8.0

## Save or verify the fixed protocol

On the first run, publish the manifests together after all checks pass. On later runs, compare their contents with the saved files. The pilot and subsequent methods read these saved IDs in order.

In [7]:
def read_contents(path):
    if path.suffix == '.json':
        return json.loads(path.read_text(encoding='utf-8'))
    with path.open(encoding='utf-8', newline='') as stream:
        return list(csv.reader(stream))

OUT.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=OUT.parent) as temp_dir:
    candidate = Path(temp_dir) / 'protocol'
    candidate.mkdir()
    for filename, frame in tables.items():
        frame.to_csv(candidate / filename, index=False)
    for filename, value in metadata_files.items():
        (candidate / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    if OUT.exists():
        changed = [f.name for f in candidate.iterdir()
                   if not (OUT / f.name).is_file() or read_contents(f) != read_contents(OUT / f.name)]
        if changed:
            raise ValueError('Frozen protocol differs. Review these files before choosing a new version: ' + ', '.join(sorted(changed)))
        print('Verified all', len(tables) + len(metadata_files), 'protocol files against the saved version.')
    else:
        candidate.rename(OUT)
        print('Saved the fixed protocol to results/protocol.')

Verified all 14 protocol files against the saved version.
